# Phase 4: Multivariate Kalman Filter on Correlated Markets

This notebook demonstrates the multivariate Kalman filter that tracks multiple
correlated prediction markets simultaneously. The key result: when one market
in a correlated cluster moves, the filter updates ALL related markets — even
before their own prices change.

## Outline
1. Generate correlated synthetic market data
2. Estimate cross-covariance with Ledoit-Wolf shrinkage
3. Run multivariate filter and show cross-market propagation
4. Compare multivariate vs independent scalar filters
5. Test with uncorrelated markets (control group)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger
logger.disable('src')

from src.filters.multivariate_kalman import MultivariateKalmanFilter
from src.filters.scalar_kalman import ScalarKalmanFilter
from src.analysis.correlation import (
    estimate_cross_covariance, correlation_matrix, compute_returns,
    sample_covariance, ledoit_wolf_shrinkage,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

rng = np.random.default_rng(42)

## 1. Generate Correlated Synthetic Markets

In [ ]:
# Simulate 3 correlated markets (like a Fed/macro cluster)
T = 500
n = 3
market_names = ['Fed Pause March', 'Fed Cut June', 'Inflation < 3%']

# True process noise covariance (markets co-move)
Q_true = np.array([
    [1e-4, 7e-5, -4e-5],
    [7e-5, 1e-4, -3e-5],
    [-4e-5, -3e-5, 8e-5],
])
R_true = np.diag([1e-3, 1.5e-3, 2e-3])

# Generate true states via correlated random walk
L_Q = np.linalg.cholesky(Q_true)
true_states = np.zeros((T, n))
true_states[0] = [0.55, 0.40, 0.60]

for t in range(1, T):
    noise = L_Q @ rng.standard_normal(n)
    true_states[t] = np.clip(true_states[t-1] + noise, 0.01, 0.99)

# Generate noisy observations
L_R = np.linalg.cholesky(R_true)
observations = np.zeros((T, n))
for t in range(T):
    obs_noise = L_R @ rng.standard_normal(n)
    observations[t] = np.clip(true_states[t] + obs_noise, 0.01, 0.99)

print(f'Generated {T} steps for {n} correlated markets')
print(f'True correlation matrix:')
print(np.round(correlation_matrix(Q_true), 3))

## 2. Estimate Cross-Covariance from Data

In [ ]:
prices = {name: observations[:, i] for i, name in enumerate(market_names)}
Q_est, market_ids = estimate_cross_covariance(prices)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# True correlation
true_corr = correlation_matrix(Q_true)
sns.heatmap(true_corr, annot=True, fmt='.3f', ax=ax1,
            xticklabels=market_names, yticklabels=market_names,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1)
ax1.set_title('True Correlation')

# Estimated correlation
est_corr = correlation_matrix(Q_est)
sns.heatmap(est_corr, annot=True, fmt='.3f', ax=ax2,
            xticklabels=market_ids, yticklabels=market_ids,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1)
ax2.set_title('Estimated Correlation (Ledoit-Wolf)')

plt.tight_layout()
plt.show()

## 3. Cross-Market Information Propagation

In [ ]:
# Run multivariate filter
R_filter = np.diag([1e-3] * n)
mkf = MultivariateKalmanFilter(n=n, Q=Q_est, R=R_filter)
result_mv = mkf.filter(observations)

# Run independent scalar filters for comparison
result_indep = np.zeros((T, n))
for j in range(n):
    skf = ScalarKalmanFilter(Q=Q_est[j, j], R=R_filter[j, j])
    r = skf.filter(observations[:, j])
    result_indep[:, j] = r.states

# Plot all markets
colors = ['#00CED1', '#FF6347', '#9370DB']
fig, axes = plt.subplots(n, 1, figsize=(14, 4*n), sharex=True)

for j in range(n):
    ax = axes[j]
    ax.plot(observations[:, j], color='gray', alpha=0.3, linewidth=0.5, label='Raw')
    ax.plot(true_states[:, j], color='green', linewidth=0.8, alpha=0.6, label='True')
    ax.plot(result_mv.states[:, j], color=colors[j], linewidth=1.5, label='Multivariate')
    ax.plot(result_indep[:, j], color=colors[j], linewidth=1.0, linestyle='--',
            alpha=0.6, label='Independent')
    ax.set_ylabel('Probability')
    ax.set_title(market_names[j])
    ax.legend(loc='best', fontsize=9)

axes[-1].set_xlabel('Time step')
fig.suptitle('Multivariate vs Independent Filters', y=1.01)
plt.tight_layout()
plt.show()

## 4. Quantify Improvement: MSE Comparison

In [ ]:
print('Mean Squared Error vs True State:')
print(f'{"Market":<25} {"Raw":>10} {"Independent":>12} {"Multivariate":>14} {"Improvement":>12}')
print('-' * 75)

for j in range(n):
    mse_raw = np.mean((observations[:, j] - true_states[:, j]) ** 2)
    mse_indep = np.mean((result_indep[:, j] - true_states[:, j]) ** 2)
    mse_mv = np.mean((result_mv.states[:, j] - true_states[:, j]) ** 2)
    improvement = (mse_indep - mse_mv) / mse_indep * 100
    print(f'{market_names[j]:<25} {mse_raw:>10.2e} {mse_indep:>12.2e} {mse_mv:>14.2e} {improvement:>11.1f}%')

## 5. Partial Observations (Missing Data)

In [ ]:
# Simulate market 1 going stale (no observations) for 50 steps
obs_partial = observations.copy()
obs_partial[200:250, 1] = np.nan  # Market 1 missing

mkf_partial = MultivariateKalmanFilter(n=n, Q=Q_est, R=R_filter)
result_partial = mkf_partial.filter(obs_partial)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(true_states[:, 1], color='green', linewidth=1.0, alpha=0.6, label='True')
ax.plot(result_partial.states[:, 1], color='#FF6347', linewidth=1.5, label='MV filter (partial obs)')
ax.axvspan(200, 250, alpha=0.1, color='red', label='Market 1 unobserved')
ax.set_xlabel('Time step')
ax.set_ylabel('Probability')
ax.set_title(f'{market_names[1]}: Estimation During Missing Data Window')
ax.legend()
plt.tight_layout()
plt.show()

# The filter continues to estimate market 1 using cross-correlations
# from the other markets that ARE still observed

## 6. Control Group: Uncorrelated Markets

In [ ]:
# Generate 3 truly independent markets
Q_indep_true = np.diag([1e-4, 1e-4, 1e-4])
L_indep = np.linalg.cholesky(Q_indep_true)

true_indep = np.zeros((T, 3))
true_indep[0] = [0.5, 0.5, 0.5]
for t in range(1, T):
    true_indep[t] = np.clip(true_indep[t-1] + L_indep @ rng.standard_normal(3), 0.01, 0.99)

obs_indep = np.clip(true_indep + rng.normal(0, 0.03, (T, 3)), 0.01, 0.99)

prices_indep = {f'Market {i}': obs_indep[:, i] for i in range(3)}
Q_est_indep, _ = estimate_cross_covariance(prices_indep)
est_corr_indep = correlation_matrix(Q_est_indep)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(est_corr_indep, annot=True, fmt='.3f', ax=ax,
            xticklabels=[f'Mkt {i}' for i in range(3)],
            yticklabels=[f'Mkt {i}' for i in range(3)],
            cmap='RdBu_r', center=0, vmin=-1, vmax=1)
ax.set_title('Estimated Correlation: Uncorrelated Markets\n(should be ~0 off-diagonal)')
plt.tight_layout()
plt.show()